# Prática — Módulo 4 – Representação Vetorial Baseada em Coocorrência

## Prática
Nesta prática o aluno constrói passo a passo diferentes representações vetoriais a partir de um pequeno corpus textual. Inicialmente são exploradas estatísticas básicas do corpus, seguida da geração de matrizes BoW e TF-IDF. Em seguida são extraídos bigramas sequenciais para construir uma matriz de coocorrência palavra-contexto. A partir dessa matriz são calculadas probabilidades conjuntas, probabilidades marginais e, por fim, os valores de PMI e PPMI, permitindo observar como relações estatísticas entre palavras emergem diretamente dos dados.

## Dataset
O corpus utilizado consiste em um conjunto pequeno de frases sintéticas envolvendo agentes (como gato, cachorro e homem), verbos de ação (bebe, come) e alimentos ou líquidos (leite, água, carne, peixe). Embora simples, esse corpus é suficiente para ilustrar como padrões de coocorrência podem revelar associações entre palavras, como verbos e objetos, permitindo visualizar o funcionamento das matrizes de coocorrência e das medidas de associação utilizadas em PLN.




In [1]:
# Importar bibliotecas
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
# Definir um pequeno corpus para analisar coocorrência entre palavras.

# Corpus minúsculo.
corpus = [
    "gato bebe agua",
    "cachorro bebe agua",
    "gato bebe leite",
    "cachorro bebe leite",
    "gato come peixe",
    "cachorro come peixe",
    "gato come carne",
    "cachorro come carne",
    "homem bebe agua",
    "homem come carne",
]

corpus

# Visualizar o corpus.
for i, frase in enumerate(corpus, 1):
    print(f"D{i}: {frase}")

D1: gato bebe agua
D2: cachorro bebe agua
D3: gato bebe leite
D4: cachorro bebe leite
D5: gato come peixe
D6: cachorro come peixe
D7: gato come carne
D8: cachorro come carne
D9: homem bebe agua
D10: homem come carne


## PARTE 1 — BoW

### Pergunta
1. Quantas palavras diferentes existem no corpus?  
Resposta: 9 palavras (agua, bebe, cachorro, carne, come, gato, homem, leite, peixe)
2. Quais palavras aparecem em mais de um documento?  
Resposta: Todas as palavras aparecem em mais de um documento.

In [3]:
# Contar palavras no corpus.
from collections import Counter

tokens = " ".join(corpus).split()

freq = Counter(tokens)
freq.most_common()


[('bebe', 5),
 ('come', 5),
 ('gato', 4),
 ('cachorro', 4),
 ('agua', 3),
 ('carne', 3),
 ('leite', 2),
 ('peixe', 2),
 ('homem', 2)]

In [4]:
# Criar matriz Bag of Words.
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(corpus)

bow = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"D{i+1}" for i in range(len(corpus))]
)

bow

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
D1,1,1,0,0,0,1,0,0,0
D2,1,1,1,0,0,0,0,0,0
D3,0,1,0,0,0,1,0,1,0
D4,0,1,1,0,0,0,0,1,0
D5,0,0,0,0,1,1,0,0,1
D6,0,0,1,0,1,0,0,0,1
D7,0,0,0,1,1,1,0,0,0
D8,0,0,1,1,1,0,0,0,0
D9,1,1,0,0,0,0,1,0,0
D10,0,0,0,1,1,0,1,0,0


### Pergunta
3. Quantos vetores existem na matriz BoW?  
Resposta: 10 vetores
4. Cada vetor representa o quê?  
Resposta: Cada vetor representa um documento individual do corpus, mapeado em um espaço vetorial onde cada posição corresponde à frequência (ou presença) de um termo específico do vocabulário.
5. Qual é a dimensão de cada vetor?  
Resposta: É um vetor de tamanho 9 (nº de colunas).
6. A dimensão do vetor depende de quê?  
Resposta: A dimensão do vetor depende do tamanho do vocabulário único do corpus (o número de features ou termos distintos), e não necessariamente do número total de palavras ou documentos.
7. Documentos que compartilham palavras terão vetores parecidos?  
Resposta: Sim. Documentos que compartilham palavras terão vetores com valores '1' (ou frequências altas) nas mesmas posições (dimensões). Analisando matematicamente, isso resulta em uma menor distância entre esses vetores no espaço vetorial.
8. Pela matriz BoW, é possível perceber que “bebe” aparece tanto com “gato” quanto com “cachorro”? Ou essa relação não fica explícita na matriz?  
Resposta: Sim, é possível observar a coocorrência dessas palavras, pois elas aparecem com valor '1' dentro do mesmo vetor (documento). No entanto, a relação semântica ou a ordem ("quem bebe o quê") não fica explícita, já que o modelo BoW desconsidera a estrutura gramatical e a vizinhança imediata das palavras (perda da sintaxe).

PARTE 2 — TF-IDF

### Pergunta
9. A matriz TF-IDF possui o mesmo número de vetores do BoW?  
Resposta: Sim, a matriz TF-IDF possui o mesmo número de vetores e a mesma dimensão que a matriz BoW, pois em ambos os métodos são mapeados o mesmo conjunto de documentos e vocabulário.
10. A dimensão dos vetores mudou em relação ao BoW?  
Resposta: Não, a dimensão permanece a mesma. Tanto no BoW quanto no TF-IDF, a dimensão do vetor é definida pelo número de termos únicos (tokens) do vocabulário do corpus.
11. Por que agora aparecem valores decimais em vez de contagens inteiras?  
Resposta: Porque em TF-IDF é aplicado uma ponderação estatística para identificar a relevância de cada termo em relação ao corpus, atribuindo um peso menor para termos muito comuns que não ajudam a diferenciar um texto do outro. Diferentemente do BoW, onde só é levado em consideração a frequência absoluta (binária).
12. Todas as palavras possuem o mesmo peso dentro de um documento?  
Resposta: Não. No TF-IDF, o peso é o produto de dois fatores: a frequência do termo no documento (TF) e o inverso da frequência nos documentos (IDF). Assim, palavras que aparecem em quase todos os documentos recebem um peso menor, enquanto palavras que aparecem com frequência em poucos documentos recebem um peso maior, pois possuem maior poder discriminativo.
13. Qual é a palavra mais informativa (mais discriminativa) de cada documento?  
Resposta: Considerando que a palavra mais informativa é aquela com maior peso TF-IDF (com menos frequência no corpus), a lista seria:

*   D1 e D2: agua;
*   D3 e D4: leite;
*   D5 e D6: peixe;
*   D7 e D8: carne;
*   D9: homem (pois aparece em apenas 2 docs, enquanto 'agua' aparece em 3);
*   D10: homem (pois aparece em apenas 2 docs, enquanto 'carne' aparece em 3).

In [5]:
# gerar matriz TF-IDF.
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(corpus)

tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"D{i+1}" for i in range(len(corpus))]
)

tfidf

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
D1,0.641771,0.512414,0.000000,0.000000,0.000000,0.570581,0.000000,0.000000,0.000000
D2,0.641771,0.512414,0.570581,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
D3,0.000000,0.482845,0.000000,0.000000,0.000000,0.537655,0.000000,0.691222,0.000000
D4,0.000000,0.482845,0.537655,0.000000,0.000000,0.000000,0.000000,0.691222,0.000000
D5,0.000000,0.000000,0.000000,0.000000,0.482845,0.537655,0.000000,0.000000,0.691222
D6,0.000000,0.000000,0.537655,0.000000,0.482845,0.000000,0.000000,0.000000,0.691222
D7,0.000000,0.000000,0.000000,0.641771,0.512414,0.570581,0.000000,0.000000,0.000000
D8,0.000000,0.000000,0.570581,0.641771,0.512414,0.000000,0.000000,0.000000,0.000000
D9,0.582818,0.465343,0.000000,0.000000,0.000000,0.000000,0.666168,0.000000,0.000000
D10,0.000000,0.000000,0.000000,0.582818,0.465343,0.000000,0.666168,0.000000,0.000000


### Pergunta
14. O TF-IDF consegue capturar relação entre palavras como bebe → leite?  
Resposta: Não, assim como o BoW, TF-IDF não consegue capturar a relação entre as palavras. O TF-IDF melhora a importância das palavras, mas não melhora o entendimento da estrutura entre elas.

## PARTE 3 — Coocorrência (n-gramas)

In [6]:
# Gerar bigramas sequenciais (janela de contexto = "1 à direita").
from nltk.util import bigrams

bigrams_list = []

for sent in corpus:
    tokens = sent.split()
    bigrams_list.extend(list(bigrams(tokens)))

bigrams_list

[('gato', 'bebe'),
 ('bebe', 'agua'),
 ('cachorro', 'bebe'),
 ('bebe', 'agua'),
 ('gato', 'bebe'),
 ('bebe', 'leite'),
 ('cachorro', 'bebe'),
 ('bebe', 'leite'),
 ('gato', 'come'),
 ('come', 'peixe'),
 ('cachorro', 'come'),
 ('come', 'peixe'),
 ('gato', 'come'),
 ('come', 'carne'),
 ('cachorro', 'come'),
 ('come', 'carne'),
 ('homem', 'bebe'),
 ('bebe', 'agua'),
 ('homem', 'come'),
 ('come', 'carne')]

In [7]:
# Contar frequência dos bigramas.
from collections import Counter

bigram_freq = Counter(bigrams_list)
bigram_freq

Counter({('gato', 'bebe'): 2,
         ('bebe', 'agua'): 3,
         ('cachorro', 'bebe'): 2,
         ('bebe', 'leite'): 2,
         ('gato', 'come'): 2,
         ('come', 'peixe'): 2,
         ('cachorro', 'come'): 2,
         ('come', 'carne'): 3,
         ('homem', 'bebe'): 1,
         ('homem', 'come'): 1})

In [8]:
# Criar matriz de coocorrência a partir dos bigramas.
words = sorted(set([w for bg in bigram_freq for w in bg]))

matrix = pd.DataFrame(0, index=words, columns=words)

for (w,c),freq in bigram_freq.items():
    matrix.loc[w,c] = freq

matrix

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,0,0,0,0,0,0,0,0,0
bebe,3,0,0,0,0,0,0,2,0
cachorro,0,2,0,0,2,0,0,0,0
carne,0,0,0,0,0,0,0,0,0
come,0,0,0,3,0,0,0,0,2
gato,0,2,0,0,2,0,0,0,0
homem,0,1,0,0,1,0,0,0,0
leite,0,0,0,0,0,0,0,0,0
peixe,0,0,0,0,0,0,0,0,0


## PARTE 4 — PPMI

### Pergunta
15. O que significa a probabilidade conjunta P(w,c) para um par de palavras?  
Resposta: A probabilidade conjunta $P(w, c)$ para um par de palavras representa a frequência relativa com que uma palavra específica ($w$) e um contexto ($c$) ocorrem simultaneamente dentro de uma janela definida no corpus. Ela indica o quão comum é a coocorrência desses dois termos no conjunto de dados.

In [9]:
# Calcular total de coocorrências.
N = matrix.values.sum()
N

np.int64(20)

In [10]:
# Calcular probabilidade conjunta dos bigramas.
N = sum(bigram_freq.values())

p_wc = {bg: freq / N for bg, freq in bigram_freq.items()}
p_wc

{('gato', 'bebe'): 0.1,
 ('bebe', 'agua'): 0.15,
 ('cachorro', 'bebe'): 0.1,
 ('bebe', 'leite'): 0.1,
 ('gato', 'come'): 0.1,
 ('come', 'peixe'): 0.1,
 ('cachorro', 'come'): 0.1,
 ('come', 'carne'): 0.15,
 ('homem', 'bebe'): 0.05,
 ('homem', 'come'): 0.05}

### Pergunta
16. Qual é o valor da soma das probabilidades conjuntas? O resultado seria o mesmo independentemente da quantidade de palavras?  
Resposta: O resultado da soma das probabilidades conjuntas $P(w, c)$ no espaço amostral é sempre 1. E sim, o resultado continuaria sendo 1 independentemente da quantidade de palavras no corpus, pois a soma de todas as probabilidades de um sistema deve representar a totalidade dos eventos possíveis (ou seja, 100%). O único ponto que muda com o aumento do número de palavras é a distribuição desses valores.

In [11]:
# Gerar matrix de probabilidade conjunta.
N = matrix.values.sum()

p_wc = matrix / N
p_wc

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0
bebe,0.15,0.00,0.0,0.00,0.00,0.0,0.0,0.1,0.0
cachorro,0.00,0.10,0.0,0.00,0.10,0.0,0.0,0.0,0.0
carne,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0
come,0.00,0.00,0.0,0.15,0.00,0.0,0.0,0.0,0.1
gato,0.00,0.10,0.0,0.00,0.10,0.0,0.0,0.0,0.0
homem,0.00,0.05,0.0,0.00,0.05,0.0,0.0,0.0,0.0
leite,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0
peixe,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0


### Pergunta
17. Se P(w,c) é alto, por si só, isso implica forte associação entre w e c?  
Resposta: Não, um $P(w,c)$ alto por si só não implica forte associação. Para confirmar a associação, deve-se comparar $P(w,c)$ com o produto das probabilidades individuais $P(w) \times P(c)$. Se a probabilidade conjunta for maior do que o esperado pelo acaso, há associação forte. Se o valor alto for apenas o reflexo da alta frequência individual das palavras (como ocorre com stop-words como 'o' ou 'de'), a associação é considerada fraca, pois a coocorrência não é informativa.
18. Como saber se duas palavras aparecem juntas mais do que seria esperado ao acaso?  
Resposta: Através da comparação entre a probabilidade conjunta $P(w, c)$ e o produto das probabilidades individuais $P(w) \times P(c)$. Se o valor da probabilidade conjunta for significativamente maior do que o produto das individuais, conclui-se que as palavras aparecem juntas com uma frequência superior à que o puro acaso explicaria.

*   Acaso: $P(w) \times P(c)$
*   Realidade: $P(w, c)$


In [13]:
# Calcular frequência marginal das palavras.
p_w = matrix.sum(axis=1) / N
p_w

,0
agua,0.00
bebe,0.25
cachorro,0.20
carne,0.00
come,0.25
gato,0.20
homem,0.10
leite,0.00
peixe,0.00


In [14]:
# Calcular frequência marginal dos contextos.
p_c = matrix.sum(axis=0) / N
p_c

,0
agua,0.15
bebe,0.25
cachorro,0.00
carne,0.15
come,0.25
gato,0.00
homem,0.00
leite,0.10
peixe,0.10


In [15]:
# Calcular PMI (Pointwise Mutual Information).
import numpy as np

pmi = np.log(p_wc / (p_w.values[:, None] * p_c.values[None, :]))
pmi = pmi.replace([np.inf, -np.inf], 0)
pmi

/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bebe,1.386294,0.000000,NaN,0.000000,0.000000,NaN,NaN,1.386294,0.000000
cachorro,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
carne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
come,0.000000,0.000000,NaN,1.386294,0.000000,NaN,NaN,0.000000,1.386294
gato,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
homem,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
leite,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
peixe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Pergunta
19. Porque apareceu NaN?  
Resposta:  O NaN ocorre em pares de palavras que possuem coocorrência zero no corpus ($P(w,c) = 0$), ou seja, palavras que nunca apareceram juntas nos documentos analisados. Como a fórmula do PMI exige o cálculo do logaritmo da probabilidade conjunta e o logaritmo de zero é matematicamente indefinido ($-\infty$), o sistema gera um erro numérico que é representado na matriz como NaN.
20. O que acontece no passo chamado Positive PMI?  
Resposta: Todos os valores negativos ou indefinidos (NaN/$-\infty$) da matriz original são substituídos por zero. Esse processo garante que a matriz contenha apenas associações positivas e relevantes, eliminando o ruído estatístico de palavras que raramente ou nunca aparecem juntas.

In [17]:
# Converter para PPMI.
ppmi = pmi.clip(lower=0).fillna(0)
ppmi

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
bebe,1.386294,0.000000,0.0,0.000000,0.000000,0.0,0.0,1.386294,0.000000
cachorro,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
carne,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
come,0.000000,0.000000,0.0,1.386294,0.000000,0.0,0.0,0.000000,1.386294
gato,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
homem,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
leite,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
peixe,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000


### Pergunta
21. A matriz PPMI indica claramente quais alimentos estão associados a bebe e quais estão associados a come. No entanto, é possível determinar a partir dela qual agente (gato, cachorro, homem) bebe ou come cada alimento específico?  
Resposta: Não necessariamente. É possível observar fortes associações entre verbos e alimentos (bebe e leite) e entre agentes e verbos (gato bebe) na matriz PPMI. Porém a estrutura sintática da frase original não é preservada, o modelo se baseia em coocorrências de pares e desconsidera a ordem das palavras. Ele não consegue distinguir quem realizou cada ação específica sobre qual alimento, especialmente em documentos onde múltiplos agentes e alimentos aparecem simultaneamente.

22. Esse tipo de identificação de associação entre verbos e objetos (por exemplo, bebe–água ou come–peixe) era possível utilizando apenas BoW ou TF-IDF? Explique por quê.  
Resposta: Não era possível de forma clara utilizando apenas BoW ou TF-IDF.  
O BoW foca apenas na contagem bruta (frequência absoluta). Por exemplo, os verbos como 'bebe' ou 'come' são muito frequentes e aparecem em quase todos os documentos, mas BoW não consegue destacar a afinidade específica entre eles e seus objetos.  
Já no TF-IDF, são penalizados as palavras que aparecem em muitos documentos. Então os verbos comuns acabam recebendo pesos muito baixos, o que dificulta a percepção de que eles possuem uma relação estatística forte com palavras específicas como 'água' ou 'peixe'.  
Essa associação só se torna evidente com modelos probabilísticos (como o PMI), que medem a coocorrência e a força da relação entre pares de termos.  
Resumidamente:

*   BoW: Diz o que tem no texto (Frequência).
*   TF-IDF: Diz o que é único no texto (Importância).
*   PMI: Diz quem anda com quem (Associação).
